# 모듈 5. 체인 구성하기 — LCEL (LangChain Expression Language)

> `|` 연산자로 Prompt → LLM → Parser를 연결하는 LCEL 파이프라인을 학습합니다. 기존 `LLMChain`을 대체하는 LangChain 1.0의 핵심 패턴입니다.

**학습 목표**
- `prompt | llm | output_parser` LCEL 기본 패턴 구성
- StrOutputParser / PydanticOutputParser를 체인에 통합
- RunnableParallel로 병렬 입력 처리 (RAG 패턴 기초)
- RunnablePassthrough로 입력 그대로 전달
- stream / batch / ainvoke 등 다양한 실행 방식 활용

In [1]:
# LangChain 1.0+ 설치
# pip install -U openai langchain langchain-core langchain-community langchain-openai
# pip install docarray tiktoken

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

# .env 파일에서 환경변수 로드
load_dotenv(find_dotenv(), override=True)

True

## 1. LCEL Basic

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI()
prompt = PromptTemplate.from_template("한국에서 꼭 먹어봐야 할 음식 3가지 말해줘")
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

response = chain.invoke({})
print(response)

1. 김치찌개: 한국인들이 매일 먹는 대표적인 김치 요리로, 김치와 돼지고기를 함께 끓여 만든 국물 요리입니다.

2. 불고기: 한국의 대표적인 요리 중 하나로, 소고기를 달궈 매운 양념에 버무려 구워 먹는 요리입니다.

3. 비빔밥: 밥 위에 다양한 채소, 고기, 계란, 나물 등을 올려 고추장을 곁들여 비벼 먹는 전통 음식으로, 맛과 영양 모두 풍부한 요리입니다.


## 2. RunnableParallel

- Langchain Expression Language (LCEL)에서 체인을 구성할 때 유용한 도우미 객체: 여러 처리 단계를 연결하거나 동시에 여러 처리를 실행할 때 사용

- 병렬 처리 - 체인 안에서 일부 입력만 그대로 다음 단계에 넘기고 싶거나 다음 입력은 처리하고, 질문 같은 건 그대로 프롬프트에 넣고 싶을 때 사용

    ```csharp
    Input: {"question": "누가 학교에서 일하나요?"}         
            │               
            ▼              
    [RunnableParallel]               
    ├─ "context" ← retriever → 유사 문서 검색          
    └─ "question" ← 그대로 전달        
            │
            ▼
    [ChatPromptTemplate] ← {context}와 {question}을 채운 프롬프트 생성
            │
            ▼
    [ChatOpenAI] → GPT 응답 생성
            │
            ▼
    [StrOutputParser] → 응답에서 텍스트만 추출
            │
            ▼
    Final Answer (string)
    ```

In [4]:
# pip install -q docarray tiktoken

In [5]:
# 1. 모듈 임포트
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

# 2. 문서 리스트 정의 (검색 대상이 될 원본 문서 = RAG)
texts = ["해리슨은 학교에서 일합니다.", "곰은 꿀을 좋아해"]

# 3. 벡터스토어 생성 및 임베딩 적용 (앞에서 정의한 문서를 숫자 벡터로 변환해 저장)
vectorstore = DocArrayInMemorySearch.from_texts(
    texts=texts,
    embedding=OpenAIEmbeddings()
)

# 4. 검색기(Retriever)로 변환 (질문과 가장 유사한 문서 찾기 기능)
retriever = vectorstore.as_retriever()

# 5. 프롬프트 템플릿 정의 (context는 검색된 문서)
template = """다음 지문에만 근거해서 질문에 답하세요:
{context} 

질문: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 6. LLM 및 출력 파서 정의
llm = ChatOpenAI()
output_parser = StrOutputParser()

# 7. 병렬 입력 준비 - 검색 & 질문 동시에 전달
setup_and_retrieval = RunnableParallel(
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
)

# 8. 체인 구성 (LCEL 파이프라인)
chain = setup_and_retrieval | prompt | llm | output_parser


C:\Users\qkrru\AppData\Local\Temp\ipykernel_19940\3947024079.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import DocArrayInMemorySearch


In [6]:
chain.invoke("헤릭슨은 어디서 일하나요?")

'답변: 학교에서 일합니다.'

In [8]:
chain.invoke("곰은 무엇을 좋아하나요?")

# 질문이 들어오면 retriever가 texts 중 올바른 문서를 검색해 context에 넣고, qeustion에는 원문 그대로 들어간다.
# context와 question 바탕으로 prompt를 완성시키고
# llm이 답변을 생성
# output_parser가 그 안에서 순수 텍스트 .content만 뽑아낸다.

'곰은 꿀을 좋아합니다.'